## Import libraries

In [3]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, dotenv_values 

load_dotenv()

INDEX_DIMENSION=1000
PINECONE_KEY=os.getenv("PINECONE_KEY")
INDEX_NAME=os.getenv("INDEX_NAME")


## EDA (tbd)

Load .csv, .json and images data

In [2]:
styles_metadata = pd.read_csv("fashion-dataset/styles.csv", delimiter=';')
images_metadata = pd.read_csv("fashion-dataset/images.csv", delimiter=';')

In [3]:
image_ids = np.sort(styles_metadata.id)
image_address = ["fashion-dataset/images/"+str(i)+".jpg" for i in image_ids]
image_address_dictionary = {k:v for k,v in zip(image_ids,image_address )}

In [4]:
image_set = {}
missed_items_ids = []
for id in image_address_dictionary:
    address = image_address_dictionary[id]
    try:
        image_set[id] = Image.open(address)
    except:
        missed_items_ids.append(id)

In [5]:
missed_items_ids

[12347, 39401, 39403, 39410, 39425]

# 1. Create Pinecone Index

In [9]:
# initialize Pinecone
pc = Pinecone(api_key=PINECONE_KEY)

In [6]:
# create index
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME, 
        dimension=INDEX_DIMENSION,
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

index = pc.Index(INDEX_NAME)
